In [ ]:
import os
import sys
import gc
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
from config import DATA_PATH


In [ ]:
fnames_probs = {
    '2024b': '2024BehaviorAgentProbabilities.csv',
    # '2025a': '2025AllAgentProbabilities.csv',
    # '2025b': '2025BehaviorAgentProbabilities.csv'
}
fnames_dfs = {
    '2024': 'dfHourly2024.csv',
    # '2025': 'dfHourly2025.csv'
}
probs = {k: pd.read_csv(os.path.join(DATA_PATH, v)) for k, v in fnames_probs.items()}
dfs = {k: pd.read_csv(os.path.join(DATA_PATH, v)) for k, v in fnames_dfs.items()}
# msgs = pd.read_csv(os.path.join(DATA_PATH, 'IHSMessagePool.csv'))

del fnames_probs, fnames_dfs
gc.collect()

# 2024

## Variables Needed
* Contextual variables 
* Intervention & Probability
    * Send vs no send: `action_id` = {1: 'Send', 0: 'No Send'}
    * Domain: `message_metric`
    * Chosen message: `core_message_id`
    * Message strategy: `strategy`
    * uniform for send vs no send: `p_send`
    * uniform for strategy & domain of chosen message (without replacement!): `p_mood`, `p_steps`, `p_sleep`
        * these are conditional probabilities P(Domain|Action=1)
* BL Characteristics

### Intervention & Probability

In [ ]:
df = probs['2024b'].copy()
p_cols = df.columns[df.columns.str.startswith('p_')]

# convert to datetime
df['recorded_time_utc'] = pd.to_datetime(df['recorded_time_utc'], utc=True)
df['decision_date'] = pd.to_datetime(df['decision_date'])

# verify decision_date reflects time in UTC
assert (df['decision_date'] == df['recorded_time_utc'].dt.date).all()

# verify Preinternship notification => before July 1
p = df['notification_id'].str.extract(r"(\D+)\d*(?:-\d+-\d)?$")[0] == 'Preinternship'
q = df['recorded_time_utc'] < pd.Timestamp("2024-07-01", tz="UTC")
assert (~p | q).all()
# verify after July 1 => not Preinternship notification
assert (q | ~p).all()

# verify NoMessage notification <=> action_id == 0
p = df['notification_id'].str.extract(r"(\D+)\d*(?:-\d+-\d)?$")[0] == 'NoMessage'
q = df['action_id'] == 0
assert (~p | q).all()
assert (~q | p).all()

# filter out preinternship
df = df.loc[df['recorded_time_utc'] >= pd.Timestamp("2024-07-01", tz="UTC")]

# verify action_id == 1 <=> core_message_id
assert (~(df['action_id'] == 1) | (~df['core_message_id'].isna())).all()
assert (df['core_message_id'].isna() | (df['action_id'] == 1)).all()

# verify probability is not missing
assert df[p_cols].notnull().all().all()

# verify there is only 1 notification per day per participant
s = df\
    .groupby('participant_id')['decision_date']\
    .apply(lambda s: s.duplicated().sum())
assert (s == 0).all()

# verify no send <=> missing metric & missing strategy
p = df['action_id'] == 0
q = df['message_metric'].isnull() & df['strategy'].isnull()
assert (~p | q).all()
assert (~q | p).all()

# verify chosen message has consistent domain and strategy
assert df\
    .groupby('core_message_id')[['message_metric', 'strategy']]\
    .nunique().eq(1).all(axis=1).all()

# verify probabilities add up to one
assert np.isclose(df[['p_send', 'p_no_send']].sum(axis=1), 1).all()
assert np.isclose(df[['p_mood', 'p_steps', 'p_sleep']].sum(axis=1), 1).all()
assert np.isclose(df[
    p_cols.drop(['p_send', 'p_no_send', 'p_mood', 'p_steps', 'p_sleep'])
].sum(axis=1), 1).all()

# verify p_send = p_no_send = 0.5
assert np.isclose(df[['p_send', 'p_no_send']], 0.5).all().all()

# subset to relevant columns
df = df[[
    'decision_id',
    'participant_id',
    'decision_date',
    'action_id',
    'message_metric',
    'strategy',
    'p_send',
    'p_mood',
    'p_steps',
    'p_sleep'
]].set_index('decision_id')

# export
df.to_csv(os.path.join(DATA_PATH, "processed/interventions_2024.csv"))

del df, p, q, s
gc.collect()

### Contextual Variables & Outcomes

In [ ]:
# column suffix -> daily aggregation (first match wins; default is mean)
RULES = {
    "steps": "sum",           # daily total
    "sleep_minutes": "sum",   # total minutes asleep
    "hr_avg": "mean",
    "hr_max": "max",
    "RHR": "median",          # usually a daily value repeated across hours
    "Mood": "mean",
    "_from_phone": "mean",    # share of hours filled from phone
}


def _rule(col: str) -> str:
    return next((how for suffix, how in RULES.items() if col.endswith(suffix)), "mean")


def aggregate(
    df: pd.DataFrame,
        keys: list[str] = ("participant_id", "device_type", "decision_date"),
    min_step_hours: int | None = None,
) -> pd.DataFrame:
    """Aggregate hourly rows to daily features, one rule per data stream.

    Adds `<col>_sd` = SD of hourly values for each hr_avg column, and
    `<col>_n` = number of non-null hours per day for each stream.
    Days with no data for a stream are NaN (not 0, which plain sum gives).
    If `min_step_hours` is set, step totals from days with fewer hours of
    step data are set to NaN, since sums from partial days are undercounts.
    """
    g = df.groupby(list(keys))
    n = g.count()
    daily = g.agg({c: _rule(c) for c in df.columns}).where(n > 0)

    if min_step_hours is not None:
        steps = [c for c in daily.columns if c.endswith("steps")]
        daily[steps] = daily[steps].where(n[steps] >= min_step_hours)

    # within-day variability of heart rate (NaN if fewer than 2 hours)
    hr = [c for c in df.columns if c.endswith("hr_avg")]
    sd = g[hr].std().add_suffix("_sd")

    counts = n[[c for c in n.columns if not c.endswith("_from_phone")]]
    return daily.join(sd).join(counts.add_suffix("_n"))


def wide2long(df, regex=r'^(Fitbit|Garmin|Healthkit)'):
    """Device columns -> long: one row per time point and device, one column per stream."""
    dev = df.filter(regex=regex)
    dev.columns = pd.MultiIndex.from_tuples(
        [c.split('_', 1) for c in dev.columns], names=['device_type', 'outcome']
    )
    return dev.stack('device_type').dropna(how='all')


df = dfs['2024'].copy()
# get date
df['time_utc'] = pd.to_datetime(df['time_utc'])
df['decision_date'] = df['time_utc'].dt.date
# filter out data before 7/1
df = df.loc[df['time_utc'] >= pd.Timestamp("2024-07-01", tz="UTC")]
# replace 0 mood with NaN
df['Mood'] = df['Mood'].replace(0, np.nan)

# convert from wide to long
idx = ['participant_id', 'decision_date', 'time_utc']
wide = df.set_index(idx)
long = wide2long(wide)
del df
gc.collect()

# combined Healthkit: watch, NaNs filled with phone
watch = long.xs('HealthkitWatch', level='device_type')
phone = long.xs('HealthkitPhone', level='device_type')
combined = (
    pd.concat({'Healthkit': watch.combine_first(phone)}, names=['device_type'])
    .reorder_levels([*idx, 'device_type'])
)
long = pd.concat([long, combined])
del watch, phone, combined
gc.collect()

# aggregate per participant, device, day; Mood is not device-specific, so aggregate it separately
daily = aggregate(long, keys=['participant_id', 'device_type', 'decision_date'])
mood = wide['Mood'].groupby(['participant_id', 'decision_date']).mean()
daily = daily.join(mood)
del wide, long, mood
gc.collect()

# export
for device_type, tmp in daily.groupby('device_type'):
    tmp.to_csv(os.path.join(DATA_PATH, f"processed/contexts_{device_type}_2024.csv"))
del tmp, device_type
gc.collect()

## Combine

In [ ]:
# TODO: iterate through device types
df1 = pd.read_csv(os.path.join(DATA_PATH, "processed/interventions_2024.csv"))\
    .set_index(['participant_id', 'decision_date'])
df2 = pd.read_csv(os.path.join(DATA_PATH, "processed/contexts_2024.csv"))\
    .set_index(['participant_id', 'decision_date'])
# TODO: determine whether join should be left or inner
df = df1.merge(df2, how='left', left_index=True, right_index=True)